# Predicting Biophysical Protein Stability

In [1]:
import pandas as pd
from Bio.Seq import Seq
from Bio.SeqUtils.ProtParam import ProteinAnalysis

# Targets and translated protein sequences from Step 4
targets_cds = {
    "MYOT (ENSSSCT00000015650)": "Gain",
    "MAPKAP1 (ENSSSCT00000040313)": "Gain",
    "ARHGEF25 (ENSSSCT00000057246)": "Loss",
    "PRPF18 (ENSSSCT00000058845)": "Gain",
    "DDX19A (ENSSSCT00000067233)": "Loss",
    "KRT12 (ENSSSCT00000041995)": "Gain",
}

# Fetch CDS sequences again or pass your translation output
import requests

stability_results = []

for tx_id, shift in [
    ("ENSSSCT00000015650", "MYOT (Gain)"),
    ("ENSSSCT00000040313", "MAPKAP1 (Gain)"),
    ("ENSSSCT00000057246", "ARHGEF25 (Loss)"),
    ("ENSSSCT00000058845", "PRPF18 (Gain)"),
    ("ENSSSCT00000067233", "DDX19A (Loss)"),
    ("ENSSSCT00000041995", "KRT12 (Gain)"),
]:

    res = requests.get(
        f"https://rest.ensembl.org/sequence/id/{tx_id}?type=cds",
        headers={"Content-Type": "application/json"},
    )
    if res.ok:
        cds_seq = res.json()["seq"]
        prot_seq = str(Seq(cds_seq).translate(to_stop=True))

        # Analyze stability metrics
        analysis = ProteinAnalysis(prot_seq)
        instability = analysis.instability_index()
        gravy = analysis.gravy()
        pi = analysis.isoelectric_point()

        stability_results.append(
            {
                "Target": shift,
                "Transcript_ID": tx_id,
                "Length_AA": len(prot_seq),
                "Instability_Index": round(instability, 2),
                "Classification": (
                    "Unstable (Short half-life)"
                    if instability > 40
                    else "Stable"
                ),
                "GRAVY_Hydrophobicity": round(gravy, 2),
                "Theoretical_pI": round(pi, 2),
            }
        )

df_stability = pd.DataFrame(stability_results)
print(df_stability.to_string(index=False))

         Target      Transcript_ID  Length_AA  Instability_Index             Classification  GRAVY_Hydrophobicity  Theoretical_pI
    MYOT (Gain) ENSSSCT00000015650        620              55.00 Unstable (Short half-life)                 -0.59            8.42
 MAPKAP1 (Gain) ENSSSCT00000040313        475              43.15 Unstable (Short half-life)                 -0.61            7.99
ARHGEF25 (Loss) ENSSSCT00000057246        580              54.73 Unstable (Short half-life)                 -0.47            5.75
  PRPF18 (Gain) ENSSSCT00000058845        319              47.19 Unstable (Short half-life)                 -0.76            8.51
  DDX19A (Loss) ENSSSCT00000067233        478              35.47                     Stable                 -0.39            6.32
   KRT12 (Gain) ENSSSCT00000041995        519              44.37 Unstable (Short half-life)                 -0.55            5.48
